# C6 — Validação Final

Abre a partição de teste uma única vez, após `G7`, `G8` e `G9` aprovados.
Reconstrói o pipeline do YAML, calibra em `val_cal` (mesmo split do C5) e avalia no teste.
No final, marca `G9_holdout` como `executado`.

**Pré-requisito:** `pip install -e .` com scikit-learn ≥ 1.6.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import re
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

try:
    import sklearn.frozen  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "Este notebook requer scikit-learn >= 1.6 (FrozenEstimator)."
    ) from exc

from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import (
    GatePendenteError,
    holdout_partition,
    load_supervised_artifact,
)
from src.evaluation.metrics import (
    metricas_completas, ic_bootstrap,
    auc, average_precision, brier as brier_metric,
)
from src.evaluation.errors import matriz_confusao, metricas_por_grupo
from src.evaluation.calibration import curva_confiabilidade, ece, mce, brier_decomposicao
from src.features.engineering import faixa_etaria as _faixa_etaria
from src.models.factory import (
    build_finalista,
    finalistas_aceitos,
    resolver_finalista,
)
from src.models.calibration import calibrar

HASH_LEN                = 16
CALIB_SPLIT_FRAC_PADRAO = 0.5
N_BINS_CALIB            = 10
MIN_N_CALIB_ANO         = 20
N_FN_AMOSTRA            = 20

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())

## Verificar gates

`G7`, `G8` e `G9` precisam estar `aprovado` antes de qualquer leitura do teste. O `load_supervised_artifact` reforça `G9_holdout` (defesa em profundidade).

In [ ]:
GATES_EXIGIDOS = ["G7_calibracao", "G8_limiar", "G9_holdout"]
status_gates   = {g: (C.cfg("gates", g) or {}).get("status") for g in GATES_EXIGIDOS}

display(pd.DataFrame({"gate": list(status_gates), "status": list(status_gates.values())}))

if faltando := [g for g, s in status_gates.items() if s != "aprovado"]:
    raise RuntimeError(f"Gates não aprovados: {faltando}. Edite configs/supervised.yaml.")

n_acessos_antes = int((C.cfg("gates", "G9_holdout") or {}).get("n_acessos") or 0)
print(f"Gates aprovados. n_acessos_antes (G9) = {n_acessos_antes}.")

## Carregar artefato e configuração congelada

A partição `teste` só é materializada porque `partitions` inclui `"teste"` **e** `G9_holdout` está aprovado. Configuração do modelo (finalista, calibração, limiar, features) precisa estar congelada no YAML.

In [ ]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()
try:
    df, manifest = load_supervised_artifact(
        ROOT_DIR,
        expected_kind=COHORT_KIND,
        partitions=("treino", "validacao", "teste"),
    )
except GatePendenteError as exc:
    raise RuntimeError(
        "G9_holdout ficou pendente entre a checagem e o carregamento. "
        "Verifique configs/supervised.yaml antes de rodar novamente."
    ) from exc

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]

selecao_cfg = C.cfg("selecao_modelos") or {}
finalista_escolhido = selecao_cfg.get("finalista_escolhido")
if finalista_escolhido is None:
    candidatos = list(selecao_cfg.get("finalistas") or [])
    if len(candidatos) != 1:
        raise RuntimeError(
            "selecao_modelos.finalista_escolhido não definido e "
            f"finalistas={candidatos}. Congele o finalista no YAML."
        )
    finalista_escolhido = candidatos[0]

calib_cfg    = C.cfg("calibracao") or {}
calib_ativa  = bool(calib_cfg.get("ativa", True))
calib_metodo = calib_cfg.get("metodo")
calib_fonte  = str(calib_cfg.get("fonte") or "validacao")

if calib_ativa and calib_metodo is None:
    raise RuntimeError("calibracao.ativa=true mas calibracao.metodo vazio no YAML.")
if calib_fonte != "validacao":
    raise RuntimeError(f"calibracao.fonte deve ser 'validacao', não {calib_fonte!r}.")

limiar_cfg     = C.cfg("limiar") or {}
limiar_final   = limiar_cfg.get("otimo")
limiar_metrica = limiar_cfg.get("metrica")
if limiar_final is None:
    raise RuntimeError("limiar.otimo vazio no YAML.")
limiar_final_val = float(limiar_final)

FEATURES = list(C.cfg("features", "final") or [])
if not FEATURES:
    raise RuntimeError("features.final vazio — congele as features antes do C6.")

proibidas = set(C.cfg("features", "proibidas") or [])
if conflito := set(FEATURES) & proibidas:
    raise RuntimeError(f"features.final contém colunas proibidas: {sorted(conflito)}")

display(pd.DataFrame([
    {"chave": "finalista",         "valor": finalista_escolhido},
    {"chave": "calibracao.metodo", "valor": calib_metodo if calib_ativa else "desativada"},
    {"chave": "limiar.otimo",      "valor": limiar_final_val},
    {"chave": "limiar.metrica",    "valor": limiar_metrica},
    {"chave": "features.final",    "valor": len(FEATURES)},
    {"chave": "n_acessos_G9",      "valor": n_acessos_antes},
]))

## Guarda contra reabertura

Se já existe manifesto C6 para o mesmo `dataset_sha256`, o teste foi aberto antes. Reabrir exige `SRAG_ALLOW_TEST_REOPEN=1` e justificativa em `docs/decisoes.md`.

In [ ]:
ALLOW_REOPEN = os.getenv("SRAG_ALLOW_TEST_REOPEN", "0") == "1"

reaberturas = []
for caminho in TABLES_DIR.glob("C6_validacao_final_*_manifesto.json"):
    try:
        dados = json.loads(caminho.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if dados.get("dataset_sha256") == manifest["dataset_sha256"]:
        reaberturas.append({"arquivo": caminho.name, "em_utc": dados.get("teste_aberto_em_utc")})

if reaberturas and not ALLOW_REOPEN:
    display(pd.DataFrame(reaberturas))
    raise RuntimeError(
        "Teste já aberto para este dataset_sha256. Defina "
        "SRAG_ALLOW_TEST_REOPEN=1 para reabrir e registe o motivo em docs/decisoes.md."
    )

print("Guarda OK." if not reaberturas else f"Reabertura autorizada ({len(reaberturas)} anterior(es)).")

## Reproduzir split do C5 e calibrar

O limiar congelado foi escolhido em `val_meas` com calibrador ajustado em `val_cal`. Localizamos o manifesto do C5 compatível e reproduzimos o split (hash de índice confirma que é o mesmo). O nome do finalista é resolvido em `src.models.factory` (aliases como `hist_gradient_boosting → boosting`); o notebook não mantém cópia do mapeamento.

In [ ]:
def _reproduzir_split_val_cal(y_val, frac_cal, seed):
    idx_cal, idx_meas = train_test_split(
        np.arange(len(y_val)),
        test_size=1.0 - frac_cal,
        random_state=seed,
        stratify=y_val.to_numpy(),
    )
    return np.asarray(idx_cal, dtype=np.int64), np.asarray(idx_meas, dtype=np.int64)


def _localizar_manifesto_c5(dataset_sha256, finalista, metodo, calib_ativa):
    for caminho in sorted(
        TABLES_DIR.glob("C5_calibracao_limiar_*_manifesto.json"),
        key=lambda p: p.stat().st_mtime, reverse=True,
    ):
        try:
            dados = json.loads(caminho.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if dados.get("dataset_sha256") != dataset_sha256:
            continue
        if calib_ativa:
            escolha = (dados.get("escolhas") or {}).get(finalista) or {}
            if escolha.get("metodo") != metodo:
                continue
        return dados, caminho
    return None, None


treino    = df.loc[df[PARTICAO] == "treino"].reset_index(drop=True)
validacao = df.loc[df[PARTICAO] == "validacao"].reset_index(drop=True)
teste     = holdout_partition(df, manifest)

X_train = treino[FEATURES]
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")
X_val   = validacao[FEATURES]
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")
X_test  = teste[FEATURES]
y_test  = pd.to_numeric(teste[ALVO], errors="coerce").astype("int8")

manifesto_c5, manifesto_c5_path = _localizar_manifesto_c5(
    manifest["dataset_sha256"], finalista_escolhido,
    calib_metodo if calib_ativa else None, calib_ativa,
)

if calib_ativa and manifesto_c5 is None:
    raise RuntimeError(
        "Nenhum manifesto C5 compatível (dataset_sha256 + metodo). "
        "Rode o C5 com a configuração atual do YAML antes do C6."
    )

HONEST_CALIB_SPLIT = bool(manifesto_c5.get("honest_calib_split", False)) if manifesto_c5 else False
CALIB_SPLIT_FRAC   = float(manifesto_c5.get("calib_split_frac", CALIB_SPLIT_FRAC_PADRAO)) if manifesto_c5 else CALIB_SPLIT_FRAC_PADRAO
CALIB_SPLIT_SEED   = int(manifesto_c5.get("calib_split_seed", C.SEED)) if manifesto_c5 else int(C.SEED)

X_val_cal, y_val_cal = X_val, y_val
if HONEST_CALIB_SPLIT:
    if manifesto_c5 is None:
        raise RuntimeError("HONEST_CALIB_SPLIT=True mas manifesto C5 não encontrado.")
    idx_cal, _ = _reproduzir_split_val_cal(y_val, CALIB_SPLIT_FRAC, CALIB_SPLIT_SEED)
    hash_local = hashlib.sha256(idx_cal.tobytes()).hexdigest()[:HASH_LEN]
    hash_c5    = manifesto_c5.get("calib_split_hash")
    if hash_c5 and hash_local != hash_c5:
        raise RuntimeError(f"Split divergente do C5 (local={hash_local}, C5={hash_c5}).")
    X_val_cal = X_val.iloc[idx_cal].reset_index(drop=True)
    y_val_cal = y_val.iloc[idx_cal].reset_index(drop=True)

if finalista_escolhido not in finalistas_aceitos():
    raise ValueError(
        f"Finalista desconhecido: {finalista_escolhido!r}. "
        f"Aceitos: {list(finalistas_aceitos())}."
    )

pipe_base = build_finalista(finalista_escolhido)
pipe_base.fit(X_train, y_train)

if calib_ativa:
    if calib_metodo is None:
        raise ValueError("calib_metodo deve ser definido quando a calibração está ativa.")
    calibrador = calibrar(pipe_base, X_val_cal, y_val_cal, method=calib_metodo)
else:
    calibrador = None

preditor = calibrador if calibrador is not None else pipe_base

display(pd.DataFrame({
    "bloco":       ["treino", "val_cal", "teste"],
    "n":           [len(treino), len(X_val_cal), len(teste)],
    "positivos":   [int(y_train.sum()), int(y_val_cal.sum()), int(y_test.sum())],
    "prevalencia": [float(y_train.mean()), float(y_val_cal.mean()), float(y_test.mean())],
}))
print(f"finalista   : {finalista_escolhido} (fábrica: {resolver_finalista(finalista_escolhido)})")
print(f"calibração  : {calib_metodo if calib_ativa else 'desativada'}")
print(f"split C5    : {'honesto' if HONEST_CALIB_SPLIT else 'validação inteira'}")

## Abrir o teste — predição única

Única execução do C6 que toca no teste. Nada aqui reajusta modelo, calibração ou limiar.

In [ ]:
abertura_teste_utc = pd.Timestamp.now(tz="UTC").isoformat()

prob_test   = preditor.predict_proba(X_test)[:, 1]
y_test_arr  = y_test.to_numpy()
y_pred_test = (prob_test >= limiar_final_val).astype(int)

if not np.isfinite(prob_test).all() or not ((prob_test >= 0) & (prob_test <= 1)).all():
    raise RuntimeError("Probabilidades inválidas no teste.")

metricas_test = metricas_completas(y_test_arr, prob_test, limiar=limiar_final_val)
matriz_test   = matriz_confusao(y_test_arr, y_pred_test)

display(pd.DataFrame([{
    "finalista":  finalista_escolhido,
    "calibracao": calib_metodo if calib_ativa else "nenhuma",
    "limiar":     limiar_final_val,
    **{k: float(v) for k, v in metricas_test.items()},
}]))
display(matriz_test)

print(f"Teste aberto em UTC: {abertura_teste_utc}")
print(f"n teste: {len(prob_test):,}  óbitos reais: {int(y_test_arr.sum()):,}")

## IC bootstrap, erros e avaliação estratificada

IC percentil para AP/AUC/Brier, falsos negativos primeiro, métricas por ano e por grupo, calibração por ano.

In [ ]:
boot_cfg   = C.cfg("avaliacao", "bootstrap") or {}
BOOT_ATIVO = bool(boot_cfg.get("ativo", True))
BOOT_REPS  = int(boot_cfg.get("repeticoes") or 1000)
BOOT_NIVEL = float(boot_cfg.get("nivel") or 0.95)
BOOT_SEED  = int(boot_cfg.get("seed") or C.SEED)

ic_resultados = {}
if BOOT_ATIVO:
    for nome, fn in [("ap", average_precision), ("auc", auc), ("brier", brier_metric)]:
        est, inf, sup = ic_bootstrap(
            y_test_arr, prob_test, metrica=fn,
            n_boot=BOOT_REPS, alpha=1.0 - BOOT_NIVEL, seed=BOOT_SEED,
        )
        ic_resultados[nome] = {
            "estimativa": est, "ic_inferior": inf, "ic_superior": sup, "nivel": BOOT_NIVEL,
        }

ic_df = pd.DataFrame([{"metrica": k, **v} for k, v in ic_resultados.items()])
display(ic_df)

In [ ]:
erros_cfg   = C.cfg("avaliacao", "analise_erros") or {}
ERROS_ATIVA = bool(erros_cfg.get("ativa", True))

df_err = teste.copy()
df_err["y_true"] = y_test_arr
df_err["y_pred"] = y_pred_test
df_err["y_prob"] = prob_test

fn = df_err[(df_err.y_true == 1) & (df_err.y_pred == 0)]
tp = df_err[(df_err.y_true == 1) & (df_err.y_pred == 1)]
fp = df_err[(df_err.y_true == 0) & (df_err.y_pred == 1)]
tn = df_err[(df_err.y_true == 0) & (df_err.y_pred == 0)]

display(pd.DataFrame([
    {"classe": "FN (óbito→cura)",  "n": len(fn), "frac": len(fn) / len(df_err)},
    {"classe": "TP (óbito→óbito)", "n": len(tp), "frac": len(tp) / len(df_err)},
    {"classe": "FP (cura→óbito)",  "n": len(fp), "frac": len(fp) / len(df_err)},
    {"classe": "TN (cura→cura)",   "n": len(tn), "frac": len(tn) / len(df_err)},
]))

comparacao_fn_tp = pd.DataFrame()
amostra_fn       = pd.DataFrame()

if ERROS_ATIVA and len(fn) > 0 and len(tp) > 0:
    feats_num = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df_err[c])]
    comparacao_fn_tp = pd.DataFrame([
        {"feature":    c,
         "fn_media":   float(fn[c].mean()),
         "tp_media":   float(tp[c].mean()),
         "delta_media": float(fn[c].mean() - tp[c].mean())}
        for c in feats_num
    ])
    comparacao_fn_tp["abs_delta"] = comparacao_fn_tp.delta_media.abs()
    comparacao_fn_tp = comparacao_fn_tp.sort_values("abs_delta", ascending=False).reset_index(drop=True)
    print("Features numéricas com maior diferença FN vs TP:")
    display(comparacao_fn_tp)

if len(fn) > 0:
    amostra_fn = fn.sort_values("y_prob", ascending=False).head(N_FN_AMOSTRA)[FEATURES + ["y_prob"]]
    print(f"Top {N_FN_AMOSTRA} falsos negativos por probabilidade predita:")
    display(amostra_fn)

In [ ]:
COLUNA_ANO = str(C.cfg("periodo", "coluna_ano") or "ano_internacao")

metricas_por_ano_df = pd.DataFrame()
curvas_por_ano      = {}
calib_por_ano_df    = pd.DataFrame()

if COLUNA_ANO in teste.columns:
    anos = sorted(teste[COLUNA_ANO].dropna().astype(int).unique().tolist())
    linhas_ano, linhas_calib = [], []

    for ano in anos:
        m = (teste[COLUNA_ANO].astype("Int64") == ano).to_numpy()
        y_a, p_a = y_test_arr[m], prob_test[m]
        if len(y_a) == 0:
            continue
        duas_classes = len(np.unique(y_a)) >= 2
        linhas_ano.append({
            "ano":        int(ano),
            "n":          int(len(y_a)),
            "positivos":  int(y_a.sum()),
            "prevalencia": float(y_a.mean()),
            "ap":   float(average_precision(y_a, p_a)) if duas_classes else float("nan"),
            "auc":  float(auc(y_a, p_a))               if duas_classes else float("nan"),
            "brier": float(brier_metric(y_a, p_a)),
            "ece":  float(ece(y_a, p_a, n_bins=N_BINS_CALIB)) if len(y_a) >= MIN_N_CALIB_ANO else float("nan"),
        })
        if len(y_a) >= MIN_N_CALIB_ANO and duas_classes:
            curvas_por_ano[int(ano)] = curva_confiabilidade(y_a, p_a, n_bins=N_BINS_CALIB)
            d = brier_decomposicao(y_a, p_a, n_bins=N_BINS_CALIB)
            linhas_calib.append({
                "ano": int(ano),
                "n":   int(len(y_a)),
                "ece": float(ece(y_a, p_a, n_bins=N_BINS_CALIB)),
                "mce": float(mce(y_a, p_a, n_bins=N_BINS_CALIB)),
                **{k: float(v) for k, v in d.items()},
            })

    metricas_por_ano_df = pd.DataFrame(linhas_ano)
    calib_por_ano_df    = pd.DataFrame(linhas_calib)

display(metricas_por_ano_df)

grupos_cfg  = list(C.cfg("avaliacao", "grupos") or [])
min_n_grupo = int(C.cfg("avaliacao", "min_n_grupo") or 100)

df_grupos = teste.copy()
if (
    "faixa_etaria" in grupos_cfg
    and "faixa_etaria" not in df_grupos.columns
    and "idade_anos" in df_grupos.columns
):
    df_grupos["faixa_etaria"] = _faixa_etaria(df_grupos["idade_anos"])

metricas_por_grupo_dict = {}
grupos_pulados = []
for grupo in grupos_cfg:
    if grupo not in df_grupos.columns:
        grupos_pulados.append(grupo)
        continue
    metricas_por_grupo_dict[grupo] = metricas_por_grupo(
        df_grupos, y_test_arr, prob_test, grupo,
        limiar=limiar_final_val, min_n=min_n_grupo,
    )
    print(f"\n=== {grupo} ===")
    display(metricas_por_grupo_dict[grupo])

if grupos_pulados:
    print(f"\nGrupos pulados (ausentes no teste): {grupos_pulados}")

display(calib_por_ano_df)

## Persistir artefatos e registar a execução

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo   = f"C6_validacao_final_{manifest['cohort_kind']}_{timestamp}"

caminho_metricas  = TABLES_DIR      / f"{prefixo}_metricas.csv"
caminho_matriz    = TABLES_DIR      / f"{prefixo}_matriz_confusao.csv"
caminho_por_ano   = TABLES_DIR      / f"{prefixo}_metricas_por_ano.csv"
caminho_calib_ano = TABLES_DIR      / f"{prefixo}_calibracao_por_ano.csv"
caminho_ic        = TABLES_DIR      / f"{prefixo}_ic_bootstrap.csv"
caminho_erros     = TABLES_DIR      / f"{prefixo}_analise_erros.csv"
caminho_fn        = TABLES_DIR      / f"{prefixo}_fn_amostra.csv"
caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_teste.parquet"
caminho_manifesto = TABLES_DIR      / f"{prefixo}_manifesto.json"

pd.DataFrame([{
    "finalista": finalista_escolhido,
    "limiar":    limiar_final_val,
    **{k: float(v) for k, v in metricas_test.items()},
}]).to_csv(caminho_metricas, index=False, encoding="utf-8-sig")
matriz_test.to_csv(caminho_matriz, encoding="utf-8-sig")
metricas_por_ano_df.to_csv(caminho_por_ano, index=False, encoding="utf-8-sig")
calib_por_ano_df.to_csv(caminho_calib_ano, index=False, encoding="utf-8-sig")
if not ic_df.empty:            ic_df.to_csv(caminho_ic, index=False, encoding="utf-8-sig")
if not comparacao_fn_tp.empty: comparacao_fn_tp.to_csv(caminho_erros, index=False, encoding="utf-8-sig")
if not amostra_fn.empty:       amostra_fn.to_csv(caminho_fn, index=False, encoding="utf-8-sig")

for grupo, tab in metricas_por_grupo_dict.items():
    tab.to_csv(TABLES_DIR / f"{prefixo}_por_grupo_{grupo}.csv", index=False, encoding="utf-8-sig")
for ano, curva in curvas_por_ano.items():
    curva.to_csv(TABLES_DIR / f"{prefixo}_curva_{ano}.csv", index=False, encoding="utf-8-sig")

pred_test = pd.DataFrame({
    "y_true":      y_test_arr,
    "y_prob":      prob_test,
    "y_pred":      y_pred_test,
    "limiar":      limiar_final_val,
    "modelo":      finalista_escolhido,
    "cohort_kind": manifest["cohort_kind"],
})
if COLUNA_ANO in teste.columns:
    pred_test[COLUNA_ANO] = teste[COLUNA_ANO].to_numpy()
pred_test.to_parquet(caminho_predicoes, index=False)

manifesto_c6 = {
    "cohort_kind":         manifest["cohort_kind"],
    "dataset_path":        manifest["dataset_path"],
    "dataset_sha256":      manifest["dataset_sha256"],
    "feature_columns":     FEATURES,
    "target_column":       ALVO,
    "partition_column":    PARTICAO,
    "n_treino":            int(len(treino)),
    "n_validacao":         int(len(validacao)),
    "n_teste":             int(len(teste)),
    "finalista_escolhido": finalista_escolhido,
    "calibracao_ativa":    calib_ativa,
    "calibracao_metodo":   calib_metodo,
    "calibracao_fonte":    calib_fonte,
    "honest_calib_split":  HONEST_CALIB_SPLIT,
    "manifesto_c5_usado":  str(manifesto_c5_path) if manifesto_c5_path else None,
    "limiar_final":        limiar_final_val,
    "limiar_metrica":      limiar_metrica,
    "teste_aberto_em_utc": abertura_teste_utc,
    "metricas_agregadas":  {k: float(v) for k, v in metricas_test.items()},
    "ic_bootstrap":        ic_resultados,
    "erros":               {"fn": len(fn), "tp": len(tp), "fp": len(fp), "tn": len(tn)},
    "grupos_avaliados":    list(metricas_por_grupo_dict),
    "grupos_pulados":      grupos_pulados,
    "artefatos": {
        "metricas":           str(caminho_metricas),
        "matriz_confusao":    str(caminho_matriz),
        "metricas_por_ano":   str(caminho_por_ano),
        "calibracao_por_ano": str(caminho_calib_ano),
        "ic_bootstrap":       str(caminho_ic),
        "analise_erros":      str(caminho_erros) if not comparacao_fn_tp.empty else None,
        "fn_amostra":         str(caminho_fn)    if not amostra_fn.empty else None,
        "predicoes_teste":    str(caminho_predicoes),
    },
}
caminho_manifesto.write_text(
    json.dumps(manifesto_c6, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

with Run(
    modelo="C6_validacao_final",
    parametros={
        "cohort_kind":         manifest["cohort_kind"],
        "finalista":           finalista_escolhido,
        "calibracao_metodo":   calib_metodo,
        "limiar_final":        limiar_final_val,
        "limiar_metrica":      limiar_metrica,
        "teste_aberto_em_utc": abertura_teste_utc,
        "n_acessos_antes":     n_acessos_antes,
    },
    dataset={"n_treino": len(treino), "n_validacao": len(validacao), "n_teste": len(teste)},
    features={"n_features": len(FEATURES), "usadas": FEATURES},
    notas="C6 — abertura única do teste. Pipeline reconstruído via build_finalista.",
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(**{k: float(v) for k, v in metricas_test.items()})

print(f"Run registrado em: {r.pasta}")

## Marcar `G9_holdout` como executado

Edição cirúrgica do YAML: só o bloco `G9_holdout` é tocado. Na próxima execução, a verificação de gates falha — disciplina de abertura única.

In [ ]:
def _marcar_g9_executado(root_dir: Path, timestamp_utc: str, n_antes: int) -> Path:
    yaml_path = root_dir / "configs" / "supervised.yaml"
    linhas    = yaml_path.read_text(encoding="utf-8").splitlines(keepends=True)

    i_ini = next(
        (i for i, ln in enumerate(linhas) if ln.rstrip() == "  G9_holdout:"),
        None,
    )
    if i_ini is None:
        raise RuntimeError("Bloco `G9_holdout:` não encontrado em supervised.yaml.")

    indent = len(linhas[i_ini]) - len(linhas[i_ini].lstrip(" "))
    i_fim  = len(linhas)
    for i in range(i_ini + 1, len(linhas)):
        ln = linhas[i]
        if ln.strip() and (len(ln) - len(ln.lstrip(" "))) < indent:
            i_fim = i
            break

    novos = {
        "status":       "executado",
        "executado_em": f'"{timestamp_utc}"',
        "n_acessos":    str(n_antes + 1),
    }
    padrao = re.compile(r"^(\s+)(\w+):\s*.*$")
    encontrados = set()
    bloco = linhas[i_ini:i_fim]
    for j, ln in enumerate(bloco):
        if (m := padrao.match(ln)) and m.group(2) in novos:
            bloco[j] = f"{m.group(1)}{m.group(2)}: {novos[m.group(2)]}\n"
            encontrados.add(m.group(2))

    if faltando := set(novos) - encontrados:
        raise RuntimeError(
            f"Campos ausentes em G9_holdout: {sorted(faltando)}. "
            "O teste JÁ FOI ABERTO — corrija o YAML e defina SRAG_ALLOW_TEST_REOPEN=1 para reexecutar."
        )

    linhas[i_ini:i_fim] = bloco
    yaml_path.write_text("".join(linhas), encoding="utf-8")
    return yaml_path


caminho_yaml = _marcar_g9_executado(ROOT_DIR, abertura_teste_utc, n_acessos_antes)

print()
print("=" * 60)
print(f"C6 concluído. AP={metricas_test['ap']:.4f}  AUC={metricas_test['auc']:.4f}  Brier={metricas_test['brier']:.4f}")
print(f"Falsos negativos: {len(fn)} de {int(y_test_arr.sum())} óbitos")
print(f"G9 marcado em {caminho_yaml}")
print("=" * 60)